In [1]:
!pip install --upgrade langchain langchain-core transformers langsmith

In [2]:
import os
from getpass import getpass

os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LangSmith Key: ")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

Enter LangSmith Key: ··········


In [3]:
import os
os.makedirs("data", exist_ok=True)

# Strong Resume
with open("data/strong_resume.txt", "w") as f:
    f.write("""Name: Rahul Sharma

Skills: Python, Machine Learning, Deep Learning, NLP, SQL, TensorFlow
Tools: Pandas, NumPy, Scikit-learn, Power BI
Experience: 3 years as Data Scientist
""")

# Average Resume
with open("data/average_resume.txt", "w") as f:
    f.write("""Name: Priya Singh

Skills: Python, SQL, Data Analysis
Tools: Excel, Pandas
Experience: 1 year as Data Analyst
""")

# Weak Resume
with open("data/weak_resume.txt", "w") as f:
    f.write("""Name: Amit Kumar

Skills: MS Word, Communication
Tools: None
Experience: Fresher
""")

# Job Description
with open("data/job_description.txt", "w") as f:
    f.write("""Looking for Data Scientist with:
- Python, Machine Learning, NLP
- TensorFlow or PyTorch
- SQL knowledge
- 2+ years experience
""")

In [4]:
!pip install langchain-community

In [9]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

In [11]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline("text-generation", model="google/flan-t5-base")

llm = HuggingFacePipeline(pipeline=pipe)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalL

In [12]:
from langchain_core.prompts import PromptTemplate

# Extract
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract skills, tools, and experience from the resume.
Do NOT assume anything.

Resume:
{resume}
"""
)

# Match
match_prompt = PromptTemplate(
    input_variables=["resume_data", "job_desc"],
    template="""
Compare resume with job description.

Return:
- matched_skills
- missing_skills
- experience_match

Resume:
{resume_data}

Job:
{job_desc}
"""
)

# Score
score_prompt = PromptTemplate(
    input_variables=["match_data"],
    template="""
Give a score from 0 to 100 based on matching.

Matching:
{match_data}
"""
)

# Explain
explain_prompt = PromptTemplate(
    input_variables=["match_data", "score"],
    template="""
Explain the score clearly.

Score: {score}

Matching:
{match_data}
"""
)

In [13]:
extract_chain = extract_prompt | llm
match_chain = match_prompt | llm
score_chain = score_prompt | llm
explain_chain = explain_prompt | llm

In [14]:
files = [
    "data/strong_resume.txt",
    "data/average_resume.txt",
    "data/weak_resume.txt"
]

job_desc = open("data/job_description.txt").read()

for file in files:
    print("\n==============================")
    print("Processing:", file)
    print("==============================")

    resume = open(file).read()

    # Step 1: Extract
    resume_data = extract_chain.invoke({"resume": resume})
    print("\n[Extracted Data]")
    print(resume_data)

    # Step 2: Match
    match_data = match_chain.invoke({
        "resume_data": resume_data,
        "job_desc": job_desc
    })
    print("\n[Matching Data]")
    print(match_data)

    # Step 3: Score
    score = score_chain.invoke({
        "match_data": match_data
    })
    print("\n[Score]")
    print(score)

    # Step 4: Explain
    explanation = explain_chain.invoke({
        "match_data": match_data,
        "score": score
    })
    print("\n[Explanation]")
    print(explanation)

    print("\n====================================\n")


Processing: data/strong_resume.txt


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Extracted Data]

Extract skills, tools, and experience from the resume.
Do NOT assume anything.

Resume:
Name: Rahul Sharma

Skills: Python, Machine Learning, Deep Learning, NLP, SQL, TensorFlow
Tools: Pandas, NumPy, Scikit-learn, Power BI
Experience: 3 years as Data Scientist

luding TensorFlow and Pandas. He has a strong interest in machine learning and data science.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Matching Data]

Compare resume with job description.

Return:
- matched_skills
- missing_skills
- experience_match

Resume:

Extract skills, tools, and experience from the resume.
Do NOT assume anything.

Resume:
Name: Rahul Sharma

Skills: Python, Machine Learning, Deep Learning, NLP, SQL, TensorFlow
Tools: Pandas, NumPy, Scikit-learn, Power BI
Experience: 3 years as Data Scientist

luding TensorFlow and Pandas. He has a strong interest in machine learning and data science.

Job:
Looking for Data Scientist with:
- Python, Machine Learning, NLP
- TensorFlow or PyTorch
- SQL knowledge
- 2+ years experience




...


[Matching Data]

Compare resume with job description.

Return:
- matched_skills
- missing_skills
- experience_match

Resume:

Extract skills, tools, and experience from the resume.
Do NOT assume anything.

Resume:
Name: Amit Kumar

Skills: MS Word, Communication
Tools: None
Experience: Fresher



Job:
Looking for Data Scientist with:
- Python, Machine Learning, NLP
- TensorFlow or PyTorch
- SQL knowledge
- 2+ years experience




Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Score]

Give a score from 0 to 100 based on matching.

Matching:

Compare resume with job description.

Return:
- matched_skills
- missing_skills
- experience_match

Resume:

Extract skills, tools, and experience from the resume.
Do NOT assume anything.

Resume:
Name: Amit Kumar

Skills: MS Word, Communication
Tools: None
Experience: Fresher



Job:
Looking for Data Scientist with:
- Python, Machine Learning, NLP
- TensorFlow or PyTorch
- SQL knowledge
- 2+ years experience





Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Explanation]

Explain the score clearly.

Score: 
Give a score from 0 to 100 based on matching.

Matching:

Compare resume with job description.

Return:
- matched_skills
- missing_skills
- experience_match

Resume:

Extract skills, tools, and experience from the resume.
Do NOT assume anything.

Resume:
Name: Amit Kumar

Skills: MS Word, Communication
Tools: None
Experience: Fresher



Job:
Looking for Data Scientist with:
- Python, Machine Learning, NLP
- TensorFlow or PyTorch
- SQL knowledge
- 2+ years experience




Matching:

Compare resume with job description.

Return:
- matched_skills
- missing_skills
- experience_match

Resume:

Extract skills, tools, and experience from the resume.
Do NOT assume anything.

Resume:
Name: Amit Kumar

Skills: MS Word, Communication
Tools: None
Experience: Fresher



Job:
Looking for Data Scientist with:
- Python, Machine Learning, NLP
- TensorFlow or PyTorch
- SQL knowledge
- 2+ years experience





